# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 00: Environment Setup

---

### What You'll Learn
- How to select the right warehouse type for different workloads
- Setting up database, schema, stages, and file formats
- Creating compute pools for container-based workloads

### Warehouse Types & Workload Alignment

| Warehouse Type | Best For | Example Workloads |
|---|---|---|
| **Standard** (XS-6XL) | General SQL queries, DML, DDL | Ad-hoc analytics, reporting, COPY INTO |
| **Standard** (Medium+) | Heavier analytics, DT refreshes | Dashboard queries, batch transforms |
| **Snowpark-Optimized** | Memory-intensive Python/Java | ML training, large DataFrames, UDFs |

> **Key Insight:** Snowpark-Optimized warehouses provide 16x more memory per node compared to Standard warehouses. Use them when your Python/Java workloads need to cache large datasets in memory.

## Step 1: Create the Database and Schema

In [ ]:
-- =============================================================================
-- DATABASE & SCHEMA SETUP
-- =============================================================================

USE ROLE ACCOUNTADMIN;

-- Create the HOL database
CREATE DATABASE IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL
    COMMENT = 'Hands-on Lab: Consumer Banking Platform covering ingestion, transformation, AI, and interoperability';

-- Create the main schema
CREATE SCHEMA IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_LAB
    COMMENT = 'Primary schema for all HOL objects';

-- Create a staging schema for intermediate objects
CREATE SCHEMA IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_STAGING
    COMMENT = 'Schema for staging and temporary objects';

USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;

## Step 2: Create Warehouses (Workload Alignment)

We create three different warehouse types to demonstrate workload alignment:

1. **HOL_XS_WH** - X-Small Standard: For lightweight operations (DDL, metadata queries, simple SELECTs)
2. **HOL_MEDIUM_WH** - Medium Standard: For analytics workloads (DT refreshes, batch ingestion, aggregations)
3. **HOL_SNOWPARK_OPT_WH** - Medium Snowpark-Optimized: For memory-intensive Python/ML workloads

In [ ]:
-- =============================================================================
-- WAREHOUSE CREATION - Demonstrating Workload Alignment
-- =============================================================================

-- 1. X-Small Standard Warehouse
--    Use case: DDL operations, metadata queries, small lookups
--    Cost: 1 credit/hour
CREATE WAREHOUSE IF NOT EXISTS HOL_XS_WH
    WAREHOUSE_SIZE = 'X-SMALL'
    WAREHOUSE_TYPE = 'STANDARD'
    AUTO_SUSPEND = 60           -- Suspend after 1 minute of inactivity
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Lightweight ops: DDL, metadata, small queries';

-- 2. Medium Standard Warehouse  
--    Use case: Analytics queries, Dynamic Table refreshes, batch COPY INTO
--    Cost: 4 credits/hour
CREATE WAREHOUSE IF NOT EXISTS HOL_MEDIUM_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    WAREHOUSE_TYPE = 'STANDARD'
    AUTO_SUSPEND = 120          -- Suspend after 2 minutes
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Analytics: DT refreshes, batch ingestion, aggregations';

-- 3. Snowpark-Optimized Warehouse
--    Use case: Python/Snowpark workloads, ML training, large DataFrame ops
--    Cost: 6 credits/hour (Medium) but with 16x more memory per node
CREATE WAREHOUSE IF NOT EXISTS HOL_SNOWPARK_OPT_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    WAREHOUSE_TYPE = 'SNOWPARK-OPTIMIZED'
    AUTO_SUSPEND = 120
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'ML/Python: Snowpark DataFrames, model training, UDFs';

In [ ]:
-- Verify warehouse creation
SHOW WAREHOUSES LIKE 'HOL_%';

## Step 3: Create Internal Stage and File Formats

We'll use an internal stage to store synthetic data in Parquet format before loading into tables.

In [ ]:
-- =============================================================================
-- STAGES & FILE FORMATS
-- =============================================================================

USE WAREHOUSE HOL_XS_WH;

-- Internal stage for Parquet files
CREATE STAGE IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_LAB.HOL_PARQUET_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Internal stage for Parquet data files';

-- Parquet file format with schema detection
CREATE FILE FORMAT IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_LAB.HOL_PARQUET_FF
    TYPE = 'PARQUET'
    COMPRESSION = 'SNAPPY';

-- CSV file format (for error table demos later)
CREATE FILE FORMAT IF NOT EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_LAB.HOL_CSV_FF
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('NULL', 'null', '');

## Step 4: Create Compute Pool (for Spark/Container workloads)

This compute pool will be used in Notebook 08 for running Apache Spark in a container to access Iceberg tables via Horizon REST Catalog.

In [ ]:
-- =============================================================================
-- COMPUTE POOL (for SPCS / Container Notebooks)
-- =============================================================================

CREATE COMPUTE POOL IF NOT EXISTS HOL_SPARK_POOL
    MIN_NODES = 1
    MAX_NODES = 2
    INSTANCE_FAMILY = CPU_X64_M
    AUTO_SUSPEND_SECS = 300
    COMMENT = 'Compute pool for Spark-on-SPCS Iceberg interoperability lab';

## Step 5: Grant Permissions

Since each participant has their own account with ACCOUNTADMIN, we ensure objects are accessible to SYSADMIN as a best practice.

In [ ]:
-- =============================================================================
-- PERMISSIONS (running as ACCOUNTADMIN in dedicated accounts)
-- =============================================================================

-- Grant usage on warehouses to SYSADMIN (best practice)
GRANT USAGE ON WAREHOUSE HOL_XS_WH TO ROLE SYSADMIN;
GRANT USAGE ON WAREHOUSE HOL_MEDIUM_WH TO ROLE SYSADMIN;
GRANT USAGE ON WAREHOUSE HOL_SNOWPARK_OPT_WH TO ROLE SYSADMIN;

-- Grant database privileges
GRANT ALL PRIVILEGES ON DATABASE JPMC_CONSUMER_BANKING_HOL TO ROLE SYSADMIN;
GRANT ALL PRIVILEGES ON ALL SCHEMAS IN DATABASE JPMC_CONSUMER_BANKING_HOL TO ROLE SYSADMIN;

-- Grant compute pool usage
GRANT USAGE ON COMPUTE POOL HOL_SPARK_POOL TO ROLE SYSADMIN;

## Step 6: Verify Setup

In [ ]:
-- =============================================================================
-- VERIFICATION
-- =============================================================================

-- Check database and schemas
SHOW SCHEMAS IN DATABASE JPMC_CONSUMER_BANKING_HOL;

-- Check stages
SHOW STAGES IN SCHEMA JPMC_CONSUMER_BANKING_HOL.HOL_LAB;

-- Check file formats  
SHOW FILE FORMATS IN SCHEMA JPMC_CONSUMER_BANKING_HOL.HOL_LAB;

-- Check compute pool
SHOW COMPUTE POOLS LIKE 'HOL_%';

## Summary

You have now set up:

| Component | Name | Purpose |
|-----------|------|---------|
| Database | `JPMC_CONSUMER_BANKING_HOL` | All HOL objects |
| Schema | `HOL_LAB` | Primary schema |
| Schema | `HOL_STAGING` | Staging/temp objects |
| Warehouse | `HOL_XS_WH` (XS Standard) | Lightweight ops |
| Warehouse | `HOL_MEDIUM_WH` (M Standard) | Analytics & DTs |
| Warehouse | `HOL_SNOWPARK_OPT_WH` (M Snowpark) | ML & Python |
| Stage | `HOL_PARQUET_STAGE` | Parquet file storage |
| File Format | `HOL_PARQUET_FF` | Parquet reader |
| File Format | `HOL_CSV_FF` | CSV reader |
| Compute Pool | `HOL_SPARK_POOL` | Spark containers |

---

**Next:** Proceed to `01_Synthetic_Data_Generation.ipynb` to create realistic consumer banking data.